# Lab 7 — Evaluator-Optimizer Agent
**Pattern: EVALUATOR-OPTIMIZER** — from Anthropic's five agentic patterns

---

## What is the Evaluator-Optimizer pattern?

Two separate agents. One generates. One scores.  
The generator never evaluates its own work. The evaluator never generates content.

The generator receives the evaluator's **specific feedback** each iteration and produces an improved version. This continues until quality threshold is met — or the budget runs out.

---

## Real-world analogy

**Code review.**

A developer writes code. A reviewer gives structured, specific feedback.  
The developer revises. Neither can do both jobs simultaneously — separation of concerns produces better output than one person doing both.

The key word is **structured**. A reviewer who says "make it better" is useless.  
A reviewer who says "the error handling in the payment function doesn't cover network timeouts" is actionable.

The **rubric** in this lab is that specific reviewer.

---

## What you will learn
1. How to build the generate → evaluate → improve loop
2. Why a structured rubric beats vague evaluation
3. How the evaluator's feedback flows back to the generator
4. Two exit conditions: quality threshold AND budget guard
5. How to track quality improvement across iterations

## Step 1 — Setup

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("✓ Client ready")

## Step 2 — The Rubric

This is the most important part of the pattern.

**Vague rubric** → vague feedback → no improvement  
**Specific rubric with named dimensions** → specific feedback → measurable improvement

Five dimensions, each scored 1–10. The evaluator must return structured JSON.  
The generator reads the `specific_feedback` field — one concrete sentence telling it exactly what to fix.

In [ ]:
RUBRIC = """
Evaluate the technical blog post on these five dimensions.
Return a JSON object with EXACTLY this structure — no other text:

{
  "scores": {
    "clarity":     <integer 1-10>,
    "accuracy":    <integer 1-10>,
    "conciseness": <integer 1-10>,
    "examples":    <integer 1-10>,
    "actionable":  <integer 1-10>
  },
  "overall": <average of the five scores, one decimal place>,
  "pass": <true if overall >= 7.5, otherwise false>,
  "weakest_dimension": "<name of the lowest-scoring dimension>",
  "specific_feedback": "<one concrete sentence: exactly what to improve and how>"
}

Scoring guide:
  clarity     — Is it understandable by someone new to the topic?
  accuracy    — Are the technical claims correct?
  conciseness — Is every sentence earning its place? No padding?
  examples    — Does it use real, concrete examples vs. vague abstractions?
  actionable  — Can the reader DO something after reading this?
"""

print("✓ Rubric defined")
print(f"  Dimensions: clarity, accuracy, conciseness, examples, actionable")
print(f"  Pass threshold: overall >= 7.5")

## Step 3 — Generator

Produces a short technical blog post excerpt.  
On iteration > 1, receives the evaluator's specific feedback and applies it directly.

> The feedback is injected into the **user message** of the next generation call.  
> The generator's system prompt stays the same — only the task changes.

In [ ]:
def generate(topic: str, feedback: str = "") -> str:
    """
    Generates a technical blog excerpt.
    feedback="" on first iteration.
    On subsequent iterations, feedback contains the evaluator's specific note.
    """
    if feedback:
        prompt = f"""Write a short technical blog post excerpt (150-200 words) on: {topic}

IMPORTANT — The previous version had this specific issue to fix:
{feedback}

Address this directly in your revised version."""
    else:
        prompt = f"Write a short technical blog post excerpt (150-200 words) on: {topic}"

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system="""You are a technical writer who makes complex engineering topics accessible.
Write clearly, use specific examples, and make every sentence count.
No fluff. No buzzwords.""",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("✓ Generator defined")

## Step 4 — Evaluator

A **separate LLM call** with a **different system prompt** — that's the architecture.  
The evaluator is optimised purely for critique. It does not generate content.

Returns structured JSON. A programmatic parser extracts the scores.

In [ ]:
def evaluate(content: str) -> dict:
    """
    Evaluates content against the rubric.
    Returns a structured dict with scores, pass/fail, weakest dimension,
    and one specific actionable feedback sentence.
    """
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=f"""You are a strict technical content evaluator.
{RUBRIC}
Return ONLY valid JSON. No preamble. No markdown fences.""",
        messages=[{"role": "user", "content": f"Evaluate this content:\n\n{content}"}]
    )

    raw = response.content[0].text.strip()
    # Strip markdown fences if the model added them despite instructions
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: force another iteration with a low score
        return {
            "scores": {k: 5 for k in ["clarity","accuracy","conciseness","examples","actionable"]},
            "overall": 5.0,
            "pass": False,
            "weakest_dimension": "unknown",
            "specific_feedback": f"Evaluator parse error. Raw: {raw[:100]}"
        }

print("✓ Evaluator defined")

## Step 5 — The Optimizer Loop

Ties generator and evaluator together.

Two exit conditions — **both** are necessary:
1. `pass == True` → quality threshold met → clean exit
2. `iteration == max_iterations` → budget guard → accept best result so far

Without condition 2, an agent that never satisfies the evaluator runs forever.

In [ ]:
def optimize(topic: str, quality_threshold: float = 7.5, max_iterations: int = 4) -> dict:
    """
    The full generate → evaluate → improve loop.
    Returns the best content found, its score, and a full iteration log.
    """
    print(f"\n── Topic: {topic[:60]}")
    print(f"── Threshold: {quality_threshold}/10  |  Max iterations: {max_iterations}\n")

    best_content  = ""
    best_score    = 0.0
    feedback      = ""
    iteration_log = []

    for iteration in range(1, max_iterations + 1):
        print(f"{'─'*50}")
        print(f"[Iteration {iteration}]")

        # GENERATE
        print("  GENERATOR → producing content...")
        content = generate(topic, feedback)
        print(f"  Preview: {content[:100]}...")

        # EVALUATE
        print("  EVALUATOR → scoring against rubric...")
        evaluation  = evaluate(content)
        score       = evaluation.get("overall", 0.0)
        passed      = evaluation.get("pass", False)
        weakest     = evaluation.get("weakest_dimension", "unknown")
        feedback    = evaluation.get("specific_feedback", "")
        scores      = evaluation.get("scores", {})

        # Display scores
        score_display = "  ".join([f"{k}={v}" for k, v in scores.items()])
        print(f"  Scores:  {score_display}")
        print(f"  Overall: {score}/10  |  Pass: {passed}")
        print(f"  Weakest: {weakest}")
        print(f"  Feedback for next iteration: {feedback}")

        iteration_log.append({
            "iteration": iteration,
            "score":     score,
            "passed":    passed,
            "weakest":   weakest,
            "feedback":  feedback
        })

        if score > best_score:
            best_score   = score
            best_content = content

        # Exit condition 1: quality threshold met
        if passed:
            print(f"\n  ✓ Threshold reached at iteration {iteration} ({score}/10)")
            break

        # Exit condition 2: budget exhausted
        if iteration == max_iterations:
            print(f"\n  ⚠ Max iterations reached. Returning best: {best_score}/10")

    return {
        "final_content":   best_content,
        "final_score":     best_score,
        "iterations_used": len(iteration_log),
        "iteration_log":   iteration_log
    }

print("✓ Optimizer loop defined")

## Step 6 — Run It

Pick a topic relevant to your bootcamp. Watch the scores evolve.

The `specific_feedback` field is the key — it's what drives improvement between iterations.

In [ ]:
result = optimize(
    topic="Why Kubernetes taints and tolerations matter for production workloads",
    quality_threshold=7.5,
    max_iterations=4
)

In [ ]:
# Final result
print("=" * 60)
print(f"FINAL SCORE: {result['final_score']}/10  |  Iterations used: {result['iterations_used']}")
print("=" * 60)
print(result["final_content"])

In [ ]:
# Score progression visualised
print("\n── Score Progression ──")
for entry in result["iteration_log"]:
    bar    = "█" * int(entry["score"])
    passed = "✓" if entry["passed"] else " "
    print(f"  [{passed}] Iteration {entry['iteration']}: {entry['score']:4.1f}/10  {bar}")

print()
print("── Feedback that drove each improvement ──")
for entry in result["iteration_log"]:
    if entry["feedback"]:
        print(f"  After iter {entry['iteration']}: {entry['feedback']}")

## Key Takeaways

| Point | Why it matters |
|-------|---------------|
| Separate generator and evaluator | One is optimised for writing. One for critique. A single agent doing both compromises both. |
| The rubric is the engineering work | Vague rubric → no improvement. Specific rubric → measurable improvement. |
| `specific_feedback` drives iteration | "Make it better" is useless. "The examples use vague abstractions — use real AWS service names" is actionable. |
| Two exit conditions are non-negotiable | Quality threshold = clean exit. Budget guard = circuit breaker. Without the budget guard, a stubborn evaluator runs your bill to infinity. |
| Score progression is measurable | You can define acceptable quality before deployment. Automated content pipelines use this exact loop. |

---

**Next lab:** Lab 8 — MCP-Style Tool Registry  
You'll build the standardised tool layer concept that MCP implements — one registry, all agents benefit.